# A3.5 · Validating what comes back

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Pass a fabricated claim through a schema check and then through a ground-truth verifier.

**Why a security engineer needs it.** An unverified claim becomes a shared premise, and a peer message is trusted more than a document it is no safer than. The control it builds is: schema validation plus an independent verifier before any claim propagates.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A tool result re-enters the context as a fact. So does a peer's message. A schema check proves the shape is right and says nothing at all about whether the claim inside it is true.

> **At CyberTravels.** The payments API returns `{"status":"refunded"}`. That is a valid shape and it is not evidence the money moved, and the advisor's hotel recommendation is the same problem in prose. R2.

## 2 · The framework

```
   tool result / peer message
            |
            v
   +------------------+   shape is valid, claim may be false
   |  schema check    |   {"status":"ok","rows":0}  <- conforms perfectly
   +--------+---------+
            v
   +------------------+   the claim, checked against something independent
   |    verifier      |   did the row actually appear in the database?
   +------------------+

   conformance is about the serialiser. accuracy is the expensive part.
```

**Mitigates: T5 Cascading Hallucination · T12 Communication Poisoning · T7 Misaligned Behaviour.**

Everything that comes back into the context is an input: tool results, peer
messages, retrieved documents, a sub-agent's summary. A1.10 and A1.12 both
happened because those inputs were trusted in proportion to how internal they
looked rather than to how checked they were.

Two different checks, and conflating them is the mistake:

**Schema validation** asks *is this the right shape*. Cheap, mechanical, catches
malformed input and injection through a field that was supposed to be an
integer. It is necessary and it proves nothing about truth — a perfectly-formed
JSON object can assert anything.

**Verification** asks *is this claim true, according to something that is true
independently of the agent*. A test that passes. A query whose result you can
re-run. A file that exists. A signature that checks.

The rule that follows: **a claim may not propagate past the hop that produced
it without a verification result attached.** Not "was it plausible" — was it
checked, by what, and what did that return.

That single field is what stops A1.12's cascade, because confidence can no
longer rise as evidence disappears: the evidence field travels with the claim,
and an empty one is visible at every hop.

It is also the answer to A1.16, which is why "ask the model whether it
succeeded" is not a verifier — it is the same component grading its own work.

> **What this control closes.**
>
> Stops a claim propagating without evidence attached. Schema validity is not truth: a well-formed object can assert anything.

## 3 · The check, as a skill

`{"status":"refunded"}` is well-formed and may be false. The skill checks four returns twice — schema, then an independent oracle — and confirms that a claim with no oracle stops as `unverifiable` rather than quietly becoming true.

### The skill — [`skills/runtime/tool-return-validation-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/tool-return-validation-check/SKILL.md)

```yaml
name: tool-return-validation-check
description: >-
  Check what a tool's return value is validated against before the agent acts on
  it — schema, then an independent oracle — and confirm that an unverifiable
  claim stops rather than propagating. Use when a tool's output becomes an
  agent's belief, or when reviewing multi-hop reasoning.
allowed-tools: Read, Grep, Glob
```

# Well-formed is not true

A tool return is untrusted input that arrives wearing the tool's authority. Two
checks are needed and they catch different things: a **schema** catches
malformed, and an **oracle** catches confidently wrong. Systems usually have the
first and treat it as if it were the second.

## When to use this

Any agent that acts on what a tool told it, and any pipeline where one step's
output is the next step's premise.

## Procedure

**1 — Define the schema per tool return.** Types and required fields. This is
the cheap check and it should be automatic; a return that fails it never
reaches the model.

**2 — Identify the oracle for each claim type.** Something independent that can
say true or false: a CVE database, a build, a test run, a second source. Not
another model — a model checking a model measures agreement, not truth.

**3 — Run the four cases.** Schema-perfect and true; schema-perfect and false;
schema-perfect with **no oracle available**; malformed. All four have to be
distinguishable in the output.

**4 — Make "unverifiable" a terminal state.** The third case is the one that
matters. A claim with no oracle must stop as `unverifiable` rather than
defaulting to true — silent promotion is how a hedge becomes a fact three hops
later.

**5 — Confirm only verified claims propagate.** Follow each case for several
hops and record which survive. The unverified ones surviving is the finding.

## Example

**Input** — the fixture committed at the top of [`scripts/tool_return_validation_check.py`](scripts/tool_return_validation_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   schema_ok=True  -> {'stopped': 'refuted', 'claim': 'libfoo has no known CVEs', 'by': 'checked against the advisory database'}
   schema_ok=True  -> {'propagated': 'test_login passes', 'verified_by': 'checked against the advisory database', 'hops': 3}
   schema_ok=True  -> {'stopped': 'unverifiable', 'claim': 'the refund was approved', 'why': 'no oracle for this claim'}
   schema_ok=False -> {'stopped': 'malformed'}

The first message is schema-perfect and confident and false. Schema
validation passed it; the oracle refuted it.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "tools": [{"name": "str", "schema": true, "oracle": "str|null"}],
  "cases": [{"claim": "str", "schema_ok": true, "oracle_verdict": "true|false|unavailable",
             "state": "verified|refuted|unverifiable|malformed", "propagated": false}],
  "unverifiable_is_terminal": true
}
```

## Failure modes

- **Treating schema conformance as verification.** It is a statement about the
  serialiser.
- **Using a model as the oracle.** Two models agreeing is not evidence.
- **Defaulting unverifiable to true** because the pipeline needs a value. That
  default is the defect.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/tool-return-validation-check/scripts/tool_return_validation_check.py
SCRIPT = "skills/runtime/tool-return-validation-check/scripts/tool_return_validation_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Four messages are checked twice. A schema-perfect, high-confidence claim is refuted by the oracle; a claim with no oracle stops with `unverifiable` rather than silently becoming true; a malformed message is caught by the schema; and only the verified claim propagates.

## Your turn

Find one place a sub-agent's output becomes another agent's input and ask what oracle checks it. If the answer is the model's own confidence, that is the component grading its own work.

---

**Next → [A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*